In [6]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
#llm = LLM(model="groq/openai/gpt-oss-20b", base_url="https://api.groq.com/openai/v1")

In [ ]:
# Setting up common configfor tool

_tool_config = dict(
    llm=dict(
        provider="ollama", # or google, openai, anthropic, llama2, ...
        config=dict(
            model="llama3.2:1b-instruct-q8_0",
            # temperature=0.5,
            # top_p=1,
            # stream=true,
        ),
    ),
    embedder=dict(
        provider="ollama", # or openai, ollama, ...
        config=dict(
            model_name="all-minilm",
            task_type="RETRIEVAL_DOCUMENT",
            # title="Embeddings",
        ),
    ),
)

#
_rag_tool_config = {
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "vectordb": {
        "provider": "chromadb",
        "config": {
            "persist_directory":"agentic-ai/chromadb",
            "allow_reset":"true",
            "is_persistent":"true",
            #"settings": Settings(persist_directory="agentic-ai/chromadb", allow_reset=True, is_persistent=True)
        }
    },
}
   

In [ ]:
from crewai_tools import CodeDocsSearchTool

@tool
def codedoc_search_tool(doc_path: str):
    """
    read api documentation and return content.
    """
    # by providing its URL:
    codedoc_search_tool = CodeDocsSearchTool(
        docs_url='https://docs.oracle.com/javase/8/docs/api/java/net/URL.html',
        config=_tool_config
    )

    #
    return codedoc_search_tool.run()

In [ ]:
import os
from crewai_tools import GithubSearchTool

#@tool
def github_search_tool(github_repo: str):
	"""
    read git hub repo and return repo, code, issues and pr.
    """
    # Initialize the tool for semantic searches within a specific GitHub repository
	_repo = 'https://github.com/brijeshdhaker'
	if github_repo :
		_repo = github_repo
	
	_git_hub_token = os.getenv("GIT_HUB_TOKEN")

	github_search_tool = GithubSearchTool(
		config=_rag_tool_config,
		#github_repo=_repo,
		gh_token=_git_hub_token,
		# Options: code, repo, pr, issue
		content_types=['repo'] 
	)

	return github_search_tool.run("docker-hadoop-cluster")

g_result = github_search_tool(github_repo='https://github.com/brijeshdhaker')
g_result

In [ ]:
from crewai_tools import DOCXSearchTool
# Instantiate Web Search Tool

#@tool
def docx_search_tool(query: str, doc_path :str):
    """
    read workd document and return document content.
    """
	# Initialize the tool for semantic searches within a specific GitHub repository
    _doc_path = '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx'
    if doc_path :
        _doc_path = doc_path

    # Initialize the tool to search within any DOCX file's content
    docx_tool = DOCXSearchTool(docx=_doc_path, config=_rag_tool_config)
    #
    return docx_tool.run(query)


#
docx_results = docx_search_tool(query="Cloudera", doc_path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx")
docx_results

In [ ]:
from crewai.tools import tool
from crewai_tools import WebsiteSearchTool

# Instantiate Web Search Tool
@tool
def web_search_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )
    #
    return search_tool.run(query)


In [ ]:
from crewai_tools import ScrapeWebsiteTool


# Instantiate Web Search Tool
@tool
def scrap_website_tool(query: str):
    """
    scrap the content of the specified website.
    """
    # Initialize the tool with the website URL, 
    # so the agent can only scrap the content of the specified website
    scrap_website_tool = ScrapeWebsiteTool(
        website_url='https://example.com',
        config=_tool_config
    )
    
    # Extract the text from the site
    #text = scrap_website_tool.run()
    #print(text)

    #
    return scrap_website_tool.run(query)

In [ ]:
# 1. Initialize the tool
from crewai_tools import PDFSearchTool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
@tool
def pdf_search_tool(query : str):
        """_summary
            Searches the pdf documents and returns results.
        """
        pdf_tool = PDFSearchTool(
            pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
            config={
                "embedding_model": {
                    "provider": "openai",
                    "config": {
                        "model": "nomic-embed-text",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "embedder": {
                    "provider": "openai",
                    "config": {
                        "model": "all-minilm",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        "persist_directory":"agentic-ai/chromadb",
                        "allow_reset":"true",
                        "is_persistent":"true",
                    }
                },
            }
        )

        return pdf_tool.run(query)

#
results = pdf_search_tool.run("Cloudera")
results

In [1]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(data)
documents

[Document(metadata={'source': 'https://docs.crewai.com/how-to/Installing-CrewAI/'}, page_content='Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agent

##### Embaddings

In [2]:
from com.example.agentic.loader.LoadManager import LoadManager
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager
from com.example.agentic.splitter.SplitManager import SplitManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading embedding model: all-MiniLM-L6-v2
Model all-MiniLM-L6-v2 loaded successfully.
Embedding dimension: 384
Generating embeddings for 3 documents...
[INFO] Generating embeddings for 3 _texts...


/home/brijeshdhaker/IdeaProjects/crewai_design_document/src/main/py/com/example/agentic/embedding/EmbeddingManager.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO] Generated Embeddings shape: (3, 384)
[INFO] Example embedding: [-2.65080370e-02 -4.21822742e-02 -4.21115160e-02 -9.99256689e-03
 -3.23159881e-02 -3.97206657e-02 -4.07359824e-02  4.74174172e-02
 -1.00659661e-01  2.52032708e-02 -1.73955094e-02 -6.81368560e-02
  4.92286012e-02  5.36423782e-03  5.64469658e-02  2.95200832e-02
  4.79517505e-02 -3.00429277e-02  4.48745117e-02 -3.20778750e-02
  6.90583745e-03  1.67928413e-02 -2.25894675e-02 -1.84403267e-02
 -6.25884235e-02  4.49555516e-02  4.92096283e-02 -3.32416296e-02
 -1.05802696e-02 -5.06255440e-02 -2.84961518e-02  9.86739025e-02
  1.04309089e-01  2.10521426e-02  6.99464381e-02  1.20269664e-01
  4.21192907e-02 -2.78327689e-02  4.01039161e-02  4.18988988e-03
  3.27943452e-03 -2.03975271e-02  2.19946131e-02 -2.73096059e-02
  2.97439545e-02 -1.99100524e-02 -4.75141592e-02 -4.04410139e-02
  8.55701044e-02  4.34455425e-02 -4.73311655e-02 -1.01990655e-01
  6.99416548e-02 -6.52870461e-02 -3.85071412e-02  1.99111346e-02
 -1.48514044e-02 -2.

#### Formatters

In [7]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key images or design about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Image with title and url for each finding",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [8]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [9]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool


@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "agentic-ai/crewai/config/agents.yaml"
    tasks_config = "agentic-ai/crewai/config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True,
            tools=[SerperDevTool()],
            llm=llm
        )
    
    @agent
    def image_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['image_researcher'],
            verbose=True,
            tools=[],
            allow_delegation=False,
            llm=llm
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'],
            output_json=ResearchOutput
        )
    
    @task
    def find_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['find_images_task'],
            output_json=ResearchImageOutput
        )
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )
		
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [10]:
from datetime import datetime

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': datetime.now().strftime('%Y-%m-%d')
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

In [11]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5096926a-aebe-4943-98a0-9f34b4c74d52                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: c7390ab7-7c22-477b-b60f-99077acf059e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find any interesting and relevant    │
│  information about Microservice Design.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='Design principles of microservices,      │
│  such as scalability, fault tolerance, and maintainability.', relevance='Highly relevant to modern software     │
│  development practices,', sources=[{'sourceTitle': 'The Microservices Edge', 'groupName': 'Edgerunners'},       │
│  {'sourceTitle': 'Software Engineering Institute (SEI)', 'groupName': 'Practices'}]),                           │
│  ResearchPoint(topic='Microservice Design Patterns', findings='Understanding key patterns such as service       │
│  composition, event sourcing, and RESTful APIs for better microservices architecture.', relevance='Important    │
│  for building scalable and maintainable microservices.', sources=[{'sourceTitle': 'DevOps Pathways',            │
│  'groupName': 'Design Patterns'}]), ResearchPoint(topic='Microservice Design Tools', findings='Exploring the    │
│  use of tools like Spring Boot, Docker, and Kubernetes for microservices development.', relevance='Useful for   │
│  developers looking to transition to a microservices-based architecture.', sources=[{'sourceTitle': 'DZone',    │
│  'groupName': 'Resources'}, {'sourceTitle': 'Microservices Patterns Forum', 'groupName': 'Discussion            │
│  Forums'}]), ResearchPoint(topic='Containerization in Microservice Design', findings='Using containerization    │
│  techniques like Docker for efficient deployment and management of microservices.', relevance='Highly relevant  │
│  to modern microservice development practices.', sources=[{'sourceTitle': 'Docker', 'groupName':                │
│  'Contribution'}]), ResearchPoint(topic='Service Meshes in Microservice Design', findings='Exploring the use    │
│  of service meshes like Istio and Linkerd for network policies, traffic management, and monitoring.',           │
│  relevance='Important for building secure and efficient microservices architectures.',                          │
│  sources=[{'sourceTitle': 'Istio Blog', 'groupName': 'Documentation'}]), ResearchPoint(topic='Microservice      │
│  Design Frameworks', findings='Using frameworks like API Design Patterns, Service Design Patterns, and          │
│  Microservices Guide for better microservice development.', relevance='Useful for developers looking to build   │
│  scalable and maintainable microservices.', sources=[{'sourceTitle': 'Design Patterns', 'groupName':            │
│  'Practices'}, {'sourceTitle': 'GitHub', 'groupName': 'Repos'}]), ResearchPoint(topic='Microservice Testing in  │
│  Agile', findings='Prioritizing testing and CI/CD pipelines for microservices development.', relevance='Highly  │
│  relevant to modern software development practices.', sources=[{'sourceTitle': 'Kubernetes', 'groupName':       │
│  'Contribution'}]), ResearchPoint(topic='Monitoring Services in Microservice Design', findings='Using           │
│  monitoring techniques like Prometheus and Grafana for real-time insights into microservices performance.',     │
│  relevance='Important for building efficient and scalable microservices architectures.',                        │
│  sources=[{'sourceTitle': 'Prometheus', 'groupName': 'Documentation'}]), ResearchPoint(topic='Microservice      │
│  Governance in Organizations', findings='Establishing policies and procedures for microservices development     │
│  within organizations.', relevance='Highly relevant to 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: reporting_task                                                                                           │
│  Agent: Microservice Design Reporting Analyst                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kn71zm1zeg49eqt9kfse2cav` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5347, Requested 5819. Please try again in 23.745s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kn71zm1zeg49eqt9kfse2cav` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5341, Requested 5819. Please try again in 23.7s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 42b6adb8-a0a0-44ce-b0a9-b1e3abcd8a3a                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/42b6adb8-a0a0-44c │
│ e-b0a9-b1e3abcd8a3a?access_code=TRACE-77a22db133                             │
│ 🔑 Access Code: TRACE-77a22db133                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 5096926a-aebe-4943-98a0-9f34b4c74d52                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

####
####

In [ ]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from crewai_tools import SerperDevTool

serper_tool = SerperDevTool(
    name="Web Search Tool",
    description="A tool to perform web searches and retrieve search results.",
    verbose=True
)

import json


@tool
def search_web_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=5, verbose=True)
    return search_tool.run(query)


# Initialize the tool to scrape the target website
scrape_tool = ScrapeWebsiteTool(website_url='https://www.designdocs.dev/')

In [ ]:
from crewai_tools import ScrapeWebsiteTool
from crewai import Agent




##### Create Tasks

##### Create Crew

In [ ]:

# Assemble the Crew
from crewai import Crew, Process

crew = Crew(
    agents=[web_researcher, image_researcher, document_writer],
    tasks=[web_search_task, image_search_task,write_task],
    #process=Process.hierarchical,
    manager_llm=llm,
    planning=True,  # Enable AI planning feature
    planning_llm=llm,
    full_output=True,
    share_crew=False,
    verbose=True
)

# Execute the Tasks
crew.kickoff()

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, FileWriterTool

# 1. Initialize Tools
search_tool = SerperDevTool()
file_writer = FileWriterTool()

# 2. Define Agents
researcher = Agent(
    role='Researcher', 
    goal='Find image URLs',
    backstory="""A design expert with a passion for sharing the best experiences in microservice desing.""", 
    tools=[search_tool],
    verbose=True,
    max_iter=3, 
    llm=llm,
    allow_delegation=False
)

writer = Agent(
    role='Writer', 
    goal='Create markdown file with images and summary', 
    backstory="""You are a solution architech and design expert with a passion for writing bleog for microservice desing.""", 
    tools=[file_writer], 
    verbose=True,
    max_iter=3, 
    llm=llm,
    allow_delegation=False
)

# 3. Define Tasks
task1 = Task(
    description="Search for images of [microservice design]",
    expected_output='List top 5 images of [microservice design]', 
    agent=researcher
)

task2 = Task(
    description="Save images as a Markdown list in 'agentic-ai/crewai/desings/project-design_images.md'",
    expected_output='A summary of the top 3 microservice desings.', 
    agent=writer, 
    output_file='agentic-ai/crewai/desings/project-design_images.md'
)

# 4. Form and Run Crew
crew = Crew(agents=[researcher, writer], tasks=[task1, task2])
crew.kickoff()